In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain.tools import tool

from langgraph.graph import (
    StateGraph,
    START,
    MessagesState
)

from langgraph.prebuilt import (
    ToolNode,
    tools_condition
)

from langchain_core.messages import HumanMessage

# ==================================================
# Fake Database
# ==================================================

students = {
    "mohit": {
        "marks": 89,
        "attendance": "92%",
        "fees": "Paid",
        "course": "Data Science"
    },

    "rahul": {
        "marks": 76,
        "attendance": "85%",
        "fees": "Pending",
        "course": "Python"
    }
}


# ==================================================
# Tools
# ==================================================

@tool
def get_marks(student_name: str):
    """
    Get marks of a student.
    """

    student = students.get(student_name.lower())

    if not student:
        return "Student not found"

    return f"{student_name}'s marks are {student['marks']}"


@tool
def get_attendance(student_name: str):
    """
    Get attendance of a student.
    """

    student = students.get(student_name.lower())

    if not student:
        return "Student not found"

    return f"{student_name}'s attendance is {student['attendance']}"


@tool
def get_fees(student_name: str):
    """
    Get fee status of a student.
    """

    student = students.get(student_name.lower())

    if not student:
        return "Student not found"

    return f"{student_name}'s fee status is {student['fees']}"


@tool
def get_student_details(student_name: str):
    """
    Get complete details of a student.
    """

    student = students.get(student_name.lower())

    if not student:
        return "Student not found"

    return student


# ==================================================
# Tool List
# ==================================================

tools = [
    get_marks,
    get_attendance,
    get_fees,
    get_student_details
]


# ==================================================
# LLM
# ==================================================

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

llm_with_tools = llm.bind_tools(tools)


# ==================================================
# Agent Node
# ==================================================

def agent_node(state: MessagesState):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


# ==================================================
# Tool Node
# ==================================================

tool_node = ToolNode(tools)


# ==================================================
# Graph
# ==================================================

builder = StateGraph(MessagesState)

builder.add_node(
    "agent",
    agent_node
)

builder.add_node(
    "tools",
    tool_node
)

builder.add_edge(
    START,
    "agent"
)

builder.add_conditional_edges(
    "agent",
    tools_condition
)

builder.add_edge(
    "tools",
    "agent"
)

graph = builder.compile()


# ==================================================
# Chat Loop
# ==================================================

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    result = graph.invoke(
        {
            "messages": [
                HumanMessage(content=query)
            ]
        }
    )

    print("\nAgent:", result["messages"][-1].content)


Agent: The student Mohit has the following details: 

- Marks: 89
- Attendance: 92%
- Fees: Paid
- Course: Data Science

Agent: Based on the information provided, here is a summary of Rahul's details:

* Marks: 76
* Attendance: 85%
* Fees: Pending

If you need more information, please let me know.

Agent: There is no need for a function call in this case, as the user has simply replied with "yes" without providing any context or requesting specific information about a student.

Agent: Here are the details of Mohit:
- Marks: 89
- Attendance: 92%
- Fee Status: Paid
- Course: Data Science
